# DQN on Grid4x4 — Fixed demand vs demand bag (1200 s)

## Why this experiment

Usual RL training on Grid4x4 replays the **same traffic file every episode**. The agent can look like it is learning while mostly **memorizing one movie**. The usual in-training “test” is often on that same file, so it does not catch this.

Here we treat it like **train vs test**:
- **Train** = demand the agent practices on
- **Test (held-out)** = unseen demand files, scored with **no learning**

| Arm | Training traffic | What it tests |
|-----|------------------|---------------|
| **Fixed** | Same `fixed_1200.rou.xml` every episode | Easy baseline (one movie) |
| **Bag** | Rotate `train_00…09.rou.xml` | Harder — must handle changing routes |
| **MaxPressure** | No RL training | Flat reference on the **same** held-out files |

- Episode length: **1200 s** (do not compare ATT to old 3600 s runs).
- Test set: `hold_00…02`, every 10 episodes for DQN; once for MaxPressure.
- Agent: **DQN**, seeds **42** and **43**, **100** episodes.

**Judge success on test (held-out) ATT**, not on train ATT alone.  
**Gap** = test − train. Positive gap ⇒ worse on unseen demand than on train.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO = Path('..').resolve()
OUT = REPO / 'data' / 'output_data' / 'tsc'
COLS = ['model', 'split', 'episode', 'travel_time', 'loss', 'reward', 'queue', 'delay', 'throughput']

RUNS = {
    ('fixed', 42): OUT / 'sumo_dqn_fixed1200' / 'sumo4x4' / 'dqn_fixed1200_s42' / 'logger',
    ('fixed', 43): OUT / 'sumo_dqn_fixed1200' / 'sumo4x4' / 'dqn_fixed1200_s43' / 'logger',
    ('bag', 42): OUT / 'sumo_dqn_bag1200' / 'sumo4x4' / 'dqn_bag1200_s42' / 'logger',
    ('bag', 43): OUT / 'sumo_dqn_bag1200' / 'sumo4x4' / 'dqn_bag1200_s43' / 'logger',
}

# MaxPressure held-out reference (same 3 hold files, no learning)
MP_LOGGER = OUT / 'sumo_maxpressure_1200' / 'sumo4x4' / 'mp_heldout_s42' / 'logger'


def load_dtl(logger_dir: Path) -> pd.DataFrame:
    paths = sorted(logger_dir.glob('*_DTL.log'))
    if not paths:
        raise FileNotFoundError(logger_dir)
    return pd.read_csv(paths[-1], sep='\t', header=None, names=COLS)


curves = []
for (condition, seed), logger_dir in RUNS.items():
    df = load_dtl(logger_dir)
    df = df.assign(condition=condition, seed=seed)
    curves.append(df)
    print(f"{condition:5s} seed={seed}  rows={len(df)}  log={logger_dir.name}")

all_df = pd.concat(curves, ignore_index=True)
train = all_df[all_df.split == 'TRAIN'].copy()
heldout = all_df[all_df.split == 'HELDOUT_MEAN'].copy()  # test curve

mp_df = load_dtl(MP_LOGGER)
mp_held = mp_df[mp_df.split == 'HELDOUT_MEAN']
MP_TEST_ATT = float(mp_held.iloc[-1].travel_time)
print('train points:', len(train), '| test (held-out) points:', len(heldout))
print(f'MaxPressure held-out mean ATT = {MP_TEST_ATT:.1f} s  (n=3 hold files)')

## 1. Test vs train (main generalization plot)

Each panel overlays **train** and **test** for one DQN condition.  
Gray dashed line = **MaxPressure** on the same held-out test set.

**What to look for:** do train/test stay close (generalizes), or does train pull ahead while test stays worse (memorizes)?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, condition, title in [
    (axes[0], 'fixed', 'Fixed — train vs test'),
    (axes[1], 'bag', 'Bag — train vs test'),
]:
    for seed, ls in [(42, '-'), (43, '--')]:
        tr = train[(train.condition == condition) & (train.seed == seed)]
        te = heldout[(heldout.condition == condition) & (heldout.seed == seed)]
        ax.plot(tr.episode, tr.travel_time, color='C0', ls=ls, alpha=0.85,
                label=f'train seed {seed}')
        ax.plot(te.episode, te.travel_time, color='C3', ls=ls, marker='o', ms=3,
                label=f'test (held-out) seed {seed}')
    ax.axhline(MP_TEST_ATT, color='gray', ls=':', lw=1.5,
               label=f'MaxPressure test ({MP_TEST_ATT:.1f}s)')
    ax.set_title(title)
    ax.set_xlabel('Episode')
    ax.set_ylabel('Average travel time (s)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle('Train vs test ATT (lower is better)', y=1.02)
fig.tight_layout()
plt.show()

### Reading this figure

- **Fixed (left):** train drops below test; late train looks best but test stays higher → better on the practiced movie than on unseen demand.
- **Bag (right):** train and test stay closer; test often ends near/below train → variety helps transfer.
- **MaxPressure line:** sanity check. Bag test sits near/slightly better than MP; fixed test is a bit worse than MP.

## 2. Compare arms: all train curves vs all test curves

Left = practice score. Right = generalization score (+ MaxPressure reference).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
style = {
    ('fixed', 42): dict(color='C0', ls='-', label='fixed seed 42'),
    ('fixed', 43): dict(color='C0', ls='--', label='fixed seed 43'),
    ('bag', 42): dict(color='C1', ls='-', label='bag seed 42'),
    ('bag', 43): dict(color='C1', ls='--', label='bag seed 43'),
}
for key, kw in style.items():
    cond, seed = key
    tr = train[(train.condition == cond) & (train.seed == seed)]
    te = heldout[(heldout.condition == cond) & (heldout.seed == seed)]
    axes[0].plot(tr.episode, tr.travel_time, **kw)
    axes[1].plot(te.episode, te.travel_time, **kw)

axes[1].axhline(MP_TEST_ATT, color='gray', ls=':', lw=1.5,
                label=f'MaxPressure test ({MP_TEST_ATT:.1f}s)')

axes[0].set_title('Train ATT (all DQN runs)')
axes[1].set_title('Test / held-out ATT (DQN + MaxPressure)')
for ax in axes:
    ax.set_xlabel('Episode')
    ax.set_ylabel('Average travel time (s)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Reading this figure

- **Train (left):** fixed (blue) ends lower than bag → fixed *looks* better if you only watch training.
- **Test (right):** bag ends lower than fixed and near MaxPressure; fixed ends above MaxPressure.
- Ranking by train ATT would pick the wrong winner for a generalization claim.

## 3. Generalization gap over training

Gap = **test ATT − train ATT**. Above zero = worse on test than on train.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for condition, color in [('fixed', 'C0'), ('bag', 'C1')]:
    for seed, ls in [(42, '-'), (43, '--')]:
        tr = train[(train.condition == condition) & (train.seed == seed)].set_index('episode')
        te = heldout[(heldout.condition == condition) & (heldout.seed == seed)].copy()
        te = te.assign(train_att=te.episode.map(tr['travel_time'])).dropna(subset=['train_att'])
        gap = te.travel_time - te.train_att
        ax.plot(te.episode, gap, color=color, ls=ls, label=f'{condition} seed {seed}')

ax.axhline(0, color='black', lw=0.8, alpha=0.5)
ax.set_title('Generalization gap (test − train)')
ax.set_xlabel('Episode')
ax.set_ylabel('ATT gap (s)')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Reading this figure

- **Fixed** ends clearly **above zero** (~+8 to +10 s): test worse than train.
- **Bag** ends **at or below zero**: test matches or beats train.
- Cleanest single picture of memorization vs transfer.

## 4. Results table (DQN + MaxPressure)

In [ ]:
rows = []
for (condition, seed), logger_dir in RUNS.items():
    df = load_dtl(logger_dir)
    tr = df[df.split == 'TRAIN'].iloc[-1]
    te = df[df.split == 'HELDOUT_MEAN'].iloc[-1]
    train_att = float(tr.travel_time)
    test_att = float(te.travel_time)
    rows.append({
        'method': 'DQN',
        'condition': condition,
        'seed': seed,
        'episodes': int(tr.episode) + 1,
        'final_train_ATT': round(train_att, 1),
        'final_test_ATT': round(test_att, 1),
        'gap (test−train)': round(test_att - train_att, 1),
        'vs_MaxPressure (test−MP)': round(test_att - MP_TEST_ATT, 1),
    })

results = pd.DataFrame(rows).sort_values(['condition', 'seed']).reset_index(drop=True)
display(results)

summary = (
    results.groupby('condition', as_index=False)
    .agg(
        mean_train_ATT=('final_train_ATT', 'mean'),
        mean_test_ATT=('final_test_ATT', 'mean'),
        mean_gap=('gap (test−train)', 'mean'),
        mean_vs_MP=('vs_MaxPressure (test−MP)', 'mean'),
    )
    .round(1)
)
print('Mean over DQN seeds:')
display(summary)

mp_row = pd.DataFrame([{
    'method': 'MaxPressure',
    'condition': 'held-out only',
    'seed': 42,
    'episodes': 1,
    'final_train_ATT': None,
    'final_test_ATT': round(MP_TEST_ATT, 1),
    'gap (test−train)': None,
    'vs_MaxPressure (test−MP)': 0.0,
}])
print('MaxPressure reference (same hold_00…02):')
display(mp_row)

## 5. Interpretation

| Method / condition | Mean train | Mean test | Gap | vs MaxPressure (test) |
|--------------------|------------|-----------|-----|------------------------|
| DQN **fixed** | ~150 s | ~159 s | **+9 s** | ~**+3 s worse** than MP |
| DQN **bag** | ~159 s | ~154 s | **−5 s** | ~**−2 s better** than MP |
| **MaxPressure** | — | **~155.8 s** | — | reference |

**What this means**

1. Fixed looks best on train but loses on test (positive gap) and lands **worse than MaxPressure** on held-out.
2. Bag looks worse on train but **wins on test**, near/slightly better than MaxPressure, with gap ≤ 0.
3. Train-only curves would wrongly crown fixed; held-out + MP is the fair scoreboard.

**Paper one-liner:** one-movie DQN overstates learning; demand-bag training + held-out eval flips the ranking and matches a strong classical baseline.

More background: `extras/DEMAND_BAG_EXPERIMENT.md`.

---

## 6. Recommended next step

**Freeze this protocol. Run PressLight the same way** (`presslight_fixed1200` vs `presslight_bag1200`, 1–2 seeds).

Why: if PressLight shows the same train-good / test-worse pattern for fixed and better held-out for bag, the claim is about the **benchmark protocol**, not “DQN is weird.”

Do **not** change bag size, episode count, or demand hardness until that twin run is done.